# LLM Training Dynamics — AI Lab Instructor Notebook

*NLP & Transformers · Hard*



10 tasks, 2 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# LLM Training Dynamics - Student Lab

This lab recreates common training-stability issues from LLM training on a small NumPy classification problem.
You will implement and compare:
- stable softmax cross-entropy
- LR warmup + cosine decay
- global-norm gradient clipping
- Adam vs AdamW
- batch-size effects

Focus: training dynamics and optimization, not building a giant transformer.

## Section 0 - Data + metrics
We use a synthetic multiclass dataset so optimizer behavior is easy to isolate.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_data(n: int = 2000, d: int = 50, k: int = 5, seed: int = 0):
    rng = np.random.default_rng(seed)
    Wtrue = rng.standard_normal((d, k))
    X = rng.standard_normal((n, d))
    logits = X @ Wtrue + 0.5 * rng.standard_normal((n, k))
    y = np.argmax(logits, axis=1).astype(np.int64)
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-8)
    perm = rng.permutation(n)
    return X[perm], y[perm]

def train_val_split(X, y, val_frac: float = 0.2):
    n = X.shape[0]
    nval = int(round(n * val_frac))
    return X[nval:], y[nval:], X[:nval], y[:nval]

X, y = make_data()
Xtr, ytr, Xva, yva = train_val_split(X, y)
n, d = Xtr.shape
k = int(y.max()) + 1
print('train', Xtr.shape, 'val', Xva.shape, 'k', k)
check('labels_ok', int(y.min()) == 0 and int(y.max()) == k - 1)

In [ ]:
# Checks
check('labels_ok', int(y.min()) == 0 and int(y.max()) == k - 1)

**Why this works.** ## Why synthetic data?
Rationale: a small synthetic dataset keeps the experiment focused on optimization behavior instead of model or dataset complexity.

## Task 2: Stable softmax + cross-entropy
*Section: Data + metrics*

### Task 0.1 - Stable softmax + cross-entropy
Implement stable `log_softmax`, cross-entropy loss, its gradient with respect to logits, and helper metrics for the linear model.

In [ ]:
# No source solution — left as an exercise.

## Task 3: Implement `lr_warmup_cosine(...)`
*Section: LR schedules (warmup + cosine decay)*

## Section 1 - LR schedules (warmup + cosine decay)

### Task 1.1
Implement `lr_warmup_cosine(step, base_lr, warmup_steps, total_steps, min_lr)`.
Warmup should start below `base_lr`; cosine decay should finish near `min_lr`.

In [ ]:
def lr_warmup_cosine(step: int, base_lr: float, warmup_steps: int, total_steps: int, min_lr: float) -> float:
    if total_steps <= 0:
        raise ValueError('total_steps must be positive')
    if warmup_steps > 0 and step < warmup_steps:
        return float(base_lr * (step + 1) / warmup_steps)
    if total_steps <= warmup_steps:
        return float(base_lr)
    span = max(1, total_steps - warmup_steps - 1)
    progress = min(max((step - warmup_steps) / span, 0.0), 1.0)
    cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
    return float(min_lr + (base_lr - min_lr) * cosine)

T = 100
base_lr = 1.0
warmup_steps = 10
min_lr = 0.1
lrs = [lr_warmup_cosine(s, base_lr, warmup_steps, T, min_lr) for s in range(T)]
print('first5', [round(v, 4) for v in lrs[:5]])
print('peak', round(max(lrs), 4), 'last', round(lrs[-1], 4))
check('warmup_increases', lrs[0] < lrs[warmup_steps - 1] <= base_lr + 1e-12)
check('floor_respected', min(lrs) >= min_lr - 1e-12)
check('last_is_min_lr', abs(lrs[-1] - min_lr) < 1e-6)

In [ ]:
# Checks
check('warmup_increases', lrs[0] < lrs[warmup_steps - 1] <= base_lr + 1e-12)
check('floor_respected', min(lrs) >= min_lr - 1e-12)
check('last_is_min_lr', abs(lrs[-1] - min_lr) < 1e-6)

**Why this works.** ## Warmup + cosine
Rationale: warmup avoids aggressive early updates, while cosine decay reduces the step size later in training.

## Task 4: Implement global-norm clipping
*Section: Gradient clipping*

## Section 2 - Gradient clipping

### Task 2.1
Implement `global_grad_norm(grads)` and `clip_global_norm(grads, max_norm)` for a gradient dict like `{"W": ..., "b": ...}`.

In [ ]:
def global_grad_norm(grads: dict) -> float:
    sq = 0.0
    for g in grads.values():
        sq += float(np.sum(g * g))
    return float(np.sqrt(sq))

def clip_global_norm(grads: dict, max_norm: float):
    raw_norm = global_grad_norm(grads)
    if raw_norm == 0.0 or raw_norm <= max_norm:
        return {k: v.copy() for k, v in grads.items()}, raw_norm, 1.0
    scale = float(max_norm / (raw_norm + 1e-12))
    clipped = {k: scale * v for k, v in grads.items()}
    return clipped, raw_norm, scale

toy_grads = {'W': np.full((3, 2), 5.0), 'b': np.array([3.0, 4.0])}
clipped, raw_norm, scale = clip_global_norm(toy_grads, max_norm=5.0)
used_norm = global_grad_norm(clipped)
print('raw_norm', round(raw_norm, 4), 'used_norm', round(used_norm, 4), 'scale', round(scale, 4))
check('raw_ge_used', raw_norm + 1e-8 >= used_norm)
check('used_bounded', used_norm <= 5.0 + 1e-8)
check('scale_valid', 0.0 < scale <= 1.0)

In [ ]:
# Checks
check('raw_ge_used', raw_norm + 1e-8 >= used_norm)
check('used_bounded', used_norm <= 5.0 + 1e-8)
check('scale_valid', 0.0 < scale <= 1.0)

**Why this works.** ## Global-norm clipping
Rationale: clip the whole gradient vector by one shared scale so the update direction stays the same but its magnitude is capped.

## Task 5: Implement SGD, Adam, and AdamW steps
*Section: Optimizers (SGD, Adam, AdamW)*

## Section 3 - Optimizers (SGD, Adam, AdamW)

### Task 3.1
Implement step functions that update `(W, b)` given gradients.
For AdamW, apply weight decay directly to the weights instead of mixing it into the adaptive gradient.

In [ ]:
# No source solution — left as an exercise.

## Task 6: Implement `train(...)` and track history
*Section: Training loop*

## Section 4 - Training loop

### Task 4.1
Implement `train(...)` so it can run different optimizers and schedules and return history.
Track: train loss, val loss, val acc, raw grad norm, used grad norm, weight norm, and lr.

In [ ]:
$$14

In [ ]:
# Checks
check('history_len', len(h['tr_loss']) == 50)
check('history_keys', set(['tr_loss', 'va_loss', 'va_acc', 'raw_grad_norm', 'grad_norm', 'w_norm', 'lr']).issubset(h.keys()))

**Why this works.** ## Training loop
Rationale: one configurable loop lets you compare schedules, clipping, batch size, and optimizer choice under controlled conditions.

## Task 7: Large LR without vs with clipping
*Section: Experiments*

## Section 5 - Experiments
Run a few controlled comparisons.

### Task 5.1
Show that a too-large constant LR can destabilize training, and clipping can cap the update magnitude.
We intentionally use a small clip threshold here so clipping visibly engages.

In [ ]:
print('A) Large LR, no clipping')
_, hA = train(Xtr, ytr, Xva, yva, steps=200, batch_size=128, base_lr=2.0, schedule='const', opt='sgd', clip_norm=None, seed=0)
print('max raw grad', round(max(hA['raw_grad_norm']), 3), 'max used grad', round(max(hA['grad_norm']), 3))
print('final tr_loss', round(hA['tr_loss'][-1], 4), 'final va_acc', round(hA['va_acc'][-1], 3))

print('B) Large LR + clipping')
_, hB = train(Xtr, ytr, Xva, yva, steps=200, batch_size=128, base_lr=2.0, schedule='const', opt='sgd', clip_norm=0.2, seed=0)
print('max raw grad', round(max(hB['raw_grad_norm']), 3), 'max used grad', round(max(hB['grad_norm']), 3))
print('final tr_loss', round(hB['tr_loss'][-1], 4), 'final va_acc', round(hB['va_acc'][-1], 3))

check('clip_caps_used_gn', max(hB['grad_norm']) <= 0.2 + 1e-6)
check('clip_not_larger_than_no_clip', max(hB['grad_norm']) <= max(hA['grad_norm']) + 1e-6)

In [ ]:
# Checks
check('clip_caps_used_gn', max(hB['grad_norm']) <= 0.2 + 1e-6)
check('clip_not_larger_than_no_clip', max(hB['grad_norm']) <= max(hA['grad_norm']) + 1e-6)

## Task 8: Constant LR vs warmup + cosine
*Section: Experiments*

### Task 5.2
Compare a constant LR schedule against warmup + cosine using SGD.

In [ ]:
print('C) Const LR')
_, hC = train(Xtr, ytr, Xva, yva, steps=200, batch_size=128, base_lr=0.5, schedule='const', opt='sgd', clip_norm=5.0, seed=0)
print('D) Warmup + cosine')
_, hD = train(Xtr, ytr, Xva, yva, steps=200, batch_size=128, base_lr=0.5, schedule='warmup_cosine', warmup_steps=20, min_lr=1e-3, opt='sgd', clip_norm=5.0, seed=0)
print('first 5 lrs D', [round(v, 4) for v in hD['lr'][:5]])
print('final va_acc C', round(hC['va_acc'][-1], 3), 'D', round(hD['va_acc'][-1], 3))
check('warmup_nonzero', min(hD['lr']) > 0.0)
check('warmup_peaks_at_base', max(hD['lr']) <= 0.5 + 1e-12)
check('warmup_starts_smaller', hD['lr'][0] < hD['lr'][19])

In [ ]:
# Checks
check('warmup_nonzero', min(hD['lr']) > 0.0)
check('warmup_peaks_at_base', max(hD['lr']) <= 0.5 + 1e-12)
check('warmup_starts_smaller', hD['lr'][0] < hD['lr'][19])

## Task 9: Adam vs AdamW weight norms
*Section: Experiments*

### Task 5.3
Compare Adam vs AdamW under the same schedule and track final weight norms.

In [ ]:
print('E) Adam (no wd)')
_, hE = train(Xtr, ytr, Xva, yva, steps=250, batch_size=128, base_lr=0.05, schedule='warmup_cosine', warmup_steps=20, min_lr=1e-3, opt='adam', clip_norm=10.0, wd=0.0, seed=0)
print('F) AdamW (wd=0.05)')
_, hF = train(Xtr, ytr, Xva, yva, steps=250, batch_size=128, base_lr=0.05, schedule='warmup_cosine', warmup_steps=20, min_lr=1e-3, opt='adamw', clip_norm=10.0, wd=0.05, seed=0)
print('final va_acc E', round(hE['va_acc'][-1], 3), 'F', round(hF['va_acc'][-1], 3))
print('final W norm E', round(hE['w_norm'][-1], 4), 'F', round(hF['w_norm'][-1], 4))
check('adamw_shrinks_weights', hF['w_norm'][-1] <= hE['w_norm'][-1] + 1e-6)

In [ ]:
# Checks
check('adamw_shrinks_weights', hF['w_norm'][-1] <= hE['w_norm'][-1] + 1e-6)

## Task 10: Small batch vs full batch
*Section: Experiments*

### Task 5.4
Compare batch size 32 against full batch under the same optimizer and schedule.

In [ ]:
print('G) Small batch (32)')
_, hG = train(Xtr, ytr, Xva, yva, steps=250, batch_size=32, base_lr=0.1, schedule='warmup_cosine', warmup_steps=20, min_lr=1e-3, opt='adamw', clip_norm=None, wd=0.02, seed=0)
print('H) Full batch')
_, hH = train(Xtr, ytr, Xva, yva, steps=250, batch_size=Xtr.shape[0], base_lr=0.1, schedule='warmup_cosine', warmup_steps=20, min_lr=1e-3, opt='adamw', clip_norm=None, wd=0.02, seed=0)
print('va_acc small', round(hG['va_acc'][-1], 3), 'full', round(hH['va_acc'][-1], 3))
print('raw_grad std small', round(float(np.std(hG['raw_grad_norm'])), 4), 'full', round(float(np.std(hH['raw_grad_norm'])), 4))
check('full_batch_smoother', float(np.std(hH['raw_grad_norm'])) <= float(np.std(hG['raw_grad_norm'])) + 1e-6)

In [ ]:
# Checks
check('full_batch_smoother', float(np.std(hH['raw_grad_norm'])) <= float(np.std(hG['raw_grad_norm'])) + 1e-6)